# mBART-50 Fine-tuning for Legal Nepali-English Translation

**Model:** facebook/mbart-large-50-many-to-many-mmt  
**Configurations:** 3 (NE→EN, EN→NE, Bidirectional)  
**Dataset:** Low-resource legal domain (~5K pairs)

## Setup
1. Runtime → **A100 GPU** + **High RAM**
2. Place `fileours.xlsx` at: `/MyDrive/Legal_NLP/fileours.xlsx`
3. Run all cells sequentially

**Models saved to:** `/content/models/` (local runtime)  
**Metrics saved to:** `/MyDrive/Legal_NLP/results/mbart/` (persistent)

---

## Cell 1: Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
print("Installing packages...")
!pip install -q torch transformers datasets sentencepiece
!pip install -q pandas numpy scikit-learn openpyxl

# Check GPU
import torch
print(f"\n{'='*60}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU! Enable GPU runtime.")
print(f"{'='*60}")

# Set paths
DRIVE_BASE = '/content/drive/MyDrive/Legal_NLP'
DATA_PATH = f'{DRIVE_BASE}/fileours.xlsx'

# Create directories
import os
os.makedirs(f"{DRIVE_BASE}/data/splits", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results/mbart/metrics", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

print(f"\n✅ Setup complete!")
print(f"Data: {DATA_PATH}")
print(f"Results: {DRIVE_BASE}/results/mbart/")
print(f"Models: /content/models/ (local runtime only)")

## Cell 2: Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re

print("="*80)
print("DATA PREPROCESSING")
print("="*80)

# Load
df = pd.read_excel(DATA_PATH)
print(f"\nLoaded: {len(df)} sentence pairs")

# Clean
def clean_text(text):
    if pd.isna(text): return ""
    return re.sub(r'\s+', ' ', str(text).strip())

df['English'] = df['English'].apply(clean_text)
df['Nepali'] = df['Nepali'].apply(clean_text)
df = df[(df['English'] != "") & (df['Nepali'] != "")]
df = df.drop_duplicates(subset=['English', 'Nepali'])

print(f"After cleaning: {len(df)} pairs")

# Split (80/10/10)
train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.111, random_state=42)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Split: {len(train_df)/len(df):.1%} / {len(val_df)/len(df):.1%} / {len(test_df)/len(df):.1%}")

# Save
train_df.to_csv(f'{DRIVE_BASE}/data/splits/train.csv', index=False)
val_df.to_csv(f'{DRIVE_BASE}/data/splits/val.csv', index=False)
test_df.to_csv(f'{DRIVE_BASE}/data/splits/test.csv', index=False)

print(f"\n✅ Data saved to: {DRIVE_BASE}/data/splits/")
print("="*80)

## Cell 3: Training Function (mBART-50)

In [ ]:
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset
import json
from datetime import datetime

def train_mbart(direction, output_dir, train_data_path, val_data_path, resume_from_checkpoint=None):
    """
    Train mBART-50 for legal translation.
    
    Args:
        direction: 'ne_en', 'en_ne', or 'bidirectional'
        output_dir: Local path to save checkpoints
        train_data_path: Path to train.csv
        val_data_path: Path to val.csv
        resume_from_checkpoint: Path to resume from (optional)
    """
    print("="*80)
    print(f"Training mBART-50: {direction.upper()}")
    print(f"Output: {output_dir}")
    print("="*80)

    # Load model and tokenizer
    model_name = "facebook/mbart-large-50-many-to-many-mmt"
    tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
    model = MBartForConditionalGeneration.from_pretrained(model_name)

    print(f"Parameters: {model.num_parameters():,}")

    # Load data
    train_df = pd.read_csv(train_data_path)
    val_df = pd.read_csv(val_data_path)

    # Language codes
    lang_codes = {'en': 'en_XX', 'ne': 'ne_NP'}

    if direction == 'ne_en':
        src_col, tgt_col = 'Nepali', 'English'
        src_lang, tgt_lang = lang_codes['ne'], lang_codes['en']
    elif direction == 'en_ne':
        src_col, tgt_col = 'English', 'Nepali'
        src_lang, tgt_lang = lang_codes['en'], lang_codes['ne']
    else:  # bidirectional
        # Create both directions
        train_ne_en = [{'source': row['Nepali'], 'target': row['English']} for _, row in train_df.iterrows()]
        train_en_ne = [{'source': row['English'], 'target': row['Nepali']} for _, row in train_df.iterrows()]
        train_data = train_ne_en + train_en_ne

        val_ne_en = [{'source': row['Nepali'], 'target': row['English']} for _, row in val_df.iterrows()]
        val_en_ne = [{'source': row['English'], 'target': row['Nepali']} for _, row in val_df.iterrows()]
        val_data = val_ne_en + val_en_ne

        train_dataset_raw = Dataset.from_list(train_data)
        val_dataset_raw = Dataset.from_list(val_data)

        tokenizer.src_lang = lang_codes['ne']  # default

        def tokenize_fn(examples):
            return tokenizer(
                examples['source'],
                text_target=examples['target'],
                padding='max_length',
                truncation=True,
                max_length=128
            )

        train_dataset = train_dataset_raw.map(tokenize_fn, batched=True, remove_columns=['source', 'target'])
        val_dataset = val_dataset_raw.map(tokenize_fn, batched=True, remove_columns=['source', 'target'])

        print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

    if direction != 'bidirectional':
        # Unidirectional training
        train_data = [{'source': row[src_col], 'target': row[tgt_col]} for _, row in train_df.iterrows()]
        val_data = [{'source': row[src_col], 'target': row[tgt_col]} for _, row in val_df.iterrows()]

        train_dataset_raw = Dataset.from_list(train_data)
        val_dataset_raw = Dataset.from_list(val_data)

        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang

        def tokenize_fn(examples):
            return tokenizer(
                examples['source'],
                text_target=examples['target'],
                padding='max_length',
                truncation=True,
                max_length=128
            )

        train_dataset = train_dataset_raw.map(tokenize_fn, batched=True, remove_columns=['source', 'target'])
        val_dataset = val_dataset_raw.map(tokenize_fn, batched=True, remove_columns=['source', 'target'])

        print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

        # Set forced decoder start token
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        model.config.forced_bos_token_id = forced_bos_token_id

    # Training arguments (optimized hyperparameters)
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=6,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=3e-5,
        warmup_steps=200,
        weight_decay=0.01,
        logging_steps=50,
        save_steps=200,
        eval_steps=200,
        eval_strategy="steps",
        save_strategy="steps",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        fp16=torch.cuda.is_available(),
        predict_with_generate=True,
        generation_max_length=128,
        seed=42,
        dataloader_num_workers=2,
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
    )

    # Data collator
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

    # Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    print("\nStarting training...")
    result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    # Save model locally
    final_path = f"{output_dir}/final_model"
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)

    # Save metrics to Drive
    metrics = {
        'model': 'mbart50',
        'direction': direction,
        'train_runtime': result.metrics.get('train_runtime', 0),
        'train_samples_per_second': result.metrics.get('train_samples_per_second', 0),
        'train_loss': result.metrics.get('train_loss', 0),
        'epoch': result.metrics.get('epoch', 0),
        'timestamp': datetime.now().isoformat()
    }

    metrics_path = f"{DRIVE_BASE}/results/mbart/metrics/mbart_{direction}_training_metrics.json"
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f"\n✅ Model saved to: {final_path} (local)")
    print(f"✅ Metrics saved to: {metrics_path} (Drive)")
    return trainer

print("✅ Training function defined")

## Cell 4: Train NE→EN

In [ ]:
output_dir = "/content/models/mbart50_ne_en"
checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {checkpoint}\n")

train_mbart('ne_en', output_dir,
            f'{DRIVE_BASE}/data/splits/train.csv',
            f'{DRIVE_BASE}/data/splits/val.csv', checkpoint)

## Cell 5: Train EN→NE

In [ ]:
output_dir = "/content/models/mbart50_en_ne"
checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {checkpoint}\n")

train_mbart('en_ne', output_dir,
            f'{DRIVE_BASE}/data/splits/train.csv',
            f'{DRIVE_BASE}/data/splits/val.csv', checkpoint)

## Cell 6: Train Bidirectional

In [ ]:
output_dir = "/content/models/mbart50_bidirectional"
checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {checkpoint}\n")

train_mbart('bidirectional', output_dir,
            f'{DRIVE_BASE}/data/splits/train.csv',
            f'{DRIVE_BASE}/data/splits/val.csv', checkpoint)

print("\n" + "="*80)
print("🎉 ALL mBART-50 TRAINING COMPLETE!")
print("="*80)
print("\nNext steps:")
print("1. Run nllb200_training.ipynb to train NLLB models")
print("2. Run results_analysis.ipynb to evaluate and generate paper figures")
print("="*80)